### String Output Parser

In [18]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm = llm)

#### TinyLlama can't provide structured output by default.
1. write 1st prompt(a detailed report)
2. write 2nd prompt(a summary of that detailed report)

In [2]:
from langchain_core.prompts import PromptTemplate

# 1st prompt
template_detailed_topic = PromptTemplate(
    template = "write a detailed report on {topic}",
    input_variables = ['topic']
)
# 2nd prompt
template_summary_topic = PromptTemplate(
    template = "Write a 5 line summary on the following text. \n {text}",
    input_variables = ['text']
) 

In [3]:
prompt_detailed = template_detailed_topic.format_prompt(
    topic =  "black hole"
)
models_output = model.invoke(prompt_detailed)
models_output

AIMessage(content='### Report on Black Holes\n\n#### Introduction\nBlack holes are some of the most mysterious and fascinating objects in the universe. They are regions in space where the gravitational pull is so strong that nothing, not even light, can escape from them. This report aims to provide a comprehensive overview of black holes, including their theoretical origins, characteristics, classification, and their role in the universe.\n\n#### Theoretical Origins\nThe concept of black holes as we understand them today evolved from the work of several scientists, most notably Karl Schwarzschild in 1916. Schwarzschild\'s solution to Einstein\'s field equations of general relativity provided the theoretical foundation for the existence of black holes. The term "black hole" was coined by John Archibald Wheeler in 1967.\n\n#### Characteristics\n1. **Singularities**: The core of a black hole is a singularity, a point of infinite density where the known laws of physics break down. The natu

In [4]:
type(models_output)

langchain_core.messages.ai.AIMessage

In [5]:
print(models_output.content)

### Report on Black Holes

#### Introduction
Black holes are some of the most mysterious and fascinating objects in the universe. They are regions in space where the gravitational pull is so strong that nothing, not even light, can escape from them. This report aims to provide a comprehensive overview of black holes, including their theoretical origins, characteristics, classification, and their role in the universe.

#### Theoretical Origins
The concept of black holes as we understand them today evolved from the work of several scientists, most notably Karl Schwarzschild in 1916. Schwarzschild's solution to Einstein's field equations of general relativity provided the theoretical foundation for the existence of black holes. The term "black hole" was coined by John Archibald Wheeler in 1967.

#### Characteristics
1. **Singularities**: The core of a black hole is a singularity, a point of infinite density where the known laws of physics break down. The nature of singularities, often des

In [6]:
prompt_summary = template_summary_topic.format_prompt(
    text = models_output.content
)
summary_output = model.invoke(prompt_summary)

In [7]:
summary_output.content

'Black holes are regions in space with immense gravitational pull, preventing anything from escaping, including light. The concept originated from Karl Schwarzschild\'s work in 1916, with the term "black hole" coined by John Archibald Wheeler in 1967. Key characteristics include singularities at their cores and event horizons marking the boundary beyond which escape is impossible. Black holes are classified into three types: stellar (5-100 solar masses), intermediate (100-100,000 solar masses), and supermassive (millions to billions of solar masses), with the latter often found at galaxy centers.'

In [8]:
# now use string output parser
from langchain_core.output_parsers import StrOutputParser

In [9]:
string_parser = StrOutputParser()

In [10]:
# now form a chain
chain = (
    template_detailed_topic|
    model|
    string_parser|
    (lambda x: {"text": x})|
    template_summary_topic|
    model|
    string_parser
)

In [11]:
output = chain.invoke({
    "topic": "sun"
})
output

"The Sun, or Sol, is the central star of the Solar System, composed mainly of hydrogen and helium and providing essential energy for life on Earth. With a diameter about 109 times that of Earth and a mass comprising 99.86% of the Solar System, the Sun has a core temperature of 15 million degrees Celsius and emits a luminosity of 384.6 quintillion megawatts. Solar activity includes sunspots, which are cooler magnetic regions, solar flares that emit radiation and affect Earth's technology, and coronal mass ejections that can cause geomagnetic storms."

### JSON output parser

In [12]:
model

ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id='Qwen/Qwen2.5-7B-Instruct', stop_sequences=[], server_kwargs={}, model_kwargs={}, model='Qwen/Qwen2.5-7B-Instruct', client=<InferenceClient(model='Qwen/Qwen2.5-7B-Instruct', timeout=120)>, async_client=<InferenceClient(model='Qwen/Qwen2.5-7B-Instruct', timeout=120)>, task='text-generation'), model_id='Qwen/Qwen2.5-7B-Instruct', temperature=0.8, top_p=0.95, max_tokens=512, model_kwargs={})

In [12]:
# define JSON parser
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()

In [13]:
# define the prompt template
template = PromptTemplate(
    template = "Give me the name, age and city of a fictional person \n {format_instruction}",
    input_variables = [], # should be an empty list
    partial_variables = {
        'format_instruction': json_parser.get_format_instructions()
    }
)

In [14]:
# make the prompt
prompt = template.format()
prompt

'Give me the name, age and city of a fictional person \n Return a JSON object.'

In [15]:
result = model.invoke(prompt)
result

AIMessage(content='```json\n{\n  "name": "Liam Chen",\n  "age": 28,\n  "city": "Shanghai"\n}\n```', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 47, 'total_tokens': 79}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cd7ac-bb56-75d3-97ba-55a238407e39-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 32, 'total_tokens': 79})

In [16]:
print(json_parser.parse(result.content))

{'name': 'Liam Chen', 'age': 28, 'city': 'Shanghai'}


In [17]:
json_result = json_parser.parse(result.content)
type(json_result)

dict

In [18]:
json_result

{'name': 'Liam Chen', 'age': 28, 'city': 'Shanghai'}

In [19]:
# using chain
json_chain = template | model | json_parser
output = json_chain.invoke({})
output

{'name': 'Liam Thompson', 'age': 28, 'city': 'Seattle'}

### Structured Output parsers

In [4]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

In [5]:
# build a schema 
schema = [
    ResponseSchema(
        name = "summary", description = "A brief summary of the review"
    ),
    ResponseSchema(
        name = "sentiment",
        description = "Return sentiment of the review either negative, positive or neutral"
    ),
    ResponseSchema(
        name = "key themes", 
        description = "Write down all the key themes inside a list"
    )
]

In [6]:
# now define the parser
structured_parser = StructuredOutputParser.from_response_schemas(
    response_schemas = schema
)

In [11]:
# build the template
from langchain_core.prompts import PromptTemplate
template = PromptTemplate(
    template = 
    """Extract structured information from the review.\n
    Review: {review_text}.
    Return ONLY a valid JSON object that strictly follows this schema: \n{format_instruction}""",
    input_variables = ['review_text'],
    partial_variables = {
        "format_instruction" : structured_parser.get_format_instructions()
    }
)

In [12]:
review_text = """
Sarah here. Absolutely love this coffee maker! Brews perfectly every time.
Super easy to clean and looks great on the counter. 
The only downside is it's quite loud and the carafe lid is flimsy.
"""

In [15]:
# Fill the template
prompt = template.format_prompt(**{
    "review_text" : review_text
})

In [16]:
prompt

StringPromptValue(text='Extract structured information from the review.\n\n    Review: \nSarah here. Absolutely love this coffee maker! Brews perfectly every time.\nSuper easy to clean and looks great on the counter. \nThe only downside is it\'s quite loud and the carafe lid is flimsy.\n.\n    Return ONLY a valid JSON object that strictly follows this schema: \nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"summary": string  // A brief summary of the review\n\t"sentiment": string  // Return sentiment of the review either negative, positive or neutral\n\t"key themes": string  // Write down all the key themes inside a list\n}\n```')

In [19]:
result = model.invoke(prompt)
result

AIMessage(content='```json\n{\n\t"summary": "Sarah loves the coffee maker for its perfect brewing and cleanability but mentions it is loud and the carafe lid is flimsy.",\n\t"sentiment": "positive",\n\t"key themes": ["brewing quality", "cleanability", "appearance", "noise level", "carafe lid quality"]\n}\n```', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 185, 'total_tokens': 260}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cd7ec-e598-72c1-9303-07981b3ba7e5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'output_tokens': 75, 'total_tokens': 260})

In [21]:
print(result.content)

```json
{
	"summary": "Sarah loves the coffee maker for its perfect brewing and cleanability but mentions it is loud and the carafe lid is flimsy.",
	"sentiment": "positive",
	"key themes": ["brewing quality", "cleanability", "appearance", "noise level", "carafe lid quality"]
}
```


In [23]:
# parse as JSON
json_result = structured_parser.parse(result.content)
json_result

{'summary': 'Sarah loves the coffee maker for its perfect brewing and cleanability but mentions it is loud and the carafe lid is flimsy.',
 'sentiment': 'positive',
 'key themes': ['brewing quality',
  'cleanability',
  'appearance',
  'noise level',
  'carafe lid quality']}

In [27]:
# with chain
chain = (
    template | model | structured_parser
)
output = chain.invoke({
    "review_text" : review_text
})

In [28]:
output

{'summary': 'The coffee maker brews perfectly and is easy to clean, but it is loud and the carafe lid is flimsy.',
 'sentiment': 'positive',
 'key themes': ['easiness of cleaning',
  'perfect brewing',
  'loud noise',
  'flimsy carafe lid']}

### PydanticOutputParser
---
**PydanticOutputParser** is a LangChain output parser that forces an LLM to return data 
matching a Pydantic model schema.

It combines:
 - LLM prompt instructions
 - Pydantic schema validation
 - automatic parsing into Python objects
This produces reliable structured outputs compared to plain text responses.

In [30]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [31]:
class Person(BaseModel):
    name: str = Field(description = "Name of the person")
    age: int = Field(
        gt = 18, # age should be greater than 18
        description = "age of the person"
    )
    city: str = Field(description = "Name of the city the person belongs to")

In [32]:
parser = PydanticOutputParser(
    pydantic_object = Person
)

In [33]:
template = PromptTemplate(
    template = "Generate the name, age and city of a fictional {place} person \n {format}",
    input_variables = ['place'],
    partial_variables = {
        'format': parser.get_format_instructions()
    }
)

In [34]:
prompt = template.invoke({
    "place": "Bangladeshi"
})
result = model.invoke(prompt)
result = parser.parse(result.content)
print(result)

name='Sakib Hossain' age=28 city='Dhaka'


In [37]:
print(template)

input_variables=['place'] input_types={} partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "Name of the city the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```'} template='Generate the name, age and city of a fictional {place} person \n {format}'


In [36]:
prompt

StringPromptValue(text='Generate the name, age and city of a fictional Bangladeshi person \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "age of the person", "exclusiveMinimum": 18, "title": "Age", "type": "integer"}, "city": {"description": "Name of the city the person belongs to", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```')

In [38]:
chain = template | model | parser 
res = chain.invoke({
    "place": "Sri Lankan"
})
res

Person(name='Chamara Senanayake', age=27, city='Colombo')